# BigDataAnalyticsPipeline — Layer 4: Analytics

Visualizes results stored in PostgreSQL after the full pipeline runs.

**Pipeline flow:**
```
Kaggle CSV + Generated 500k rows
  → Layer 1: Python Ingestion
  → Layer 2: PySpark Processing
  → Layer 3: PostgreSQL Storage
  → Layer 4: This Notebook ✅
```

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import psycopg2
from dotenv import load_dotenv

# Load .env from project root
load_dotenv(os.path.join(os.path.dirname(os.getcwd()), ".env"))
load_dotenv(os.path.join(os.getcwd(), ".env"))  # fallback if running from project root

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

print("Libraries loaded.")

## 1. Connect to PostgreSQL

In [ ]:
def get_connection():
    return psycopg2.connect(
        host=os.getenv("DB_HOST", "localhost"),
        port=int(os.getenv("DB_PORT", 5432)),
        dbname=os.getenv("DB_NAME", "bigdata_pipeline"),
        user=os.getenv("DB_USER", "postgres"),
        password=os.getenv("DB_PASSWORD"),
    )

conn = get_connection()
print("Connected to PostgreSQL:", os.getenv("DB_NAME"))

## 2. Load Tables from DB

In [ ]:
df_revenue   = pd.read_sql("SELECT * FROM revenue_per_day ORDER BY purchase_date", conn)
df_products  = pd.read_sql("SELECT * FROM top_products ORDER BY total_revenue DESC", conn)
df_customers = pd.read_sql("SELECT * FROM customer_segmentation", conn)

conn.close()

print(f"revenue_per_day:      {len(df_revenue):,} rows")
print(f"top_products:         {len(df_products):,} rows")
print(f"customer_segmentation:{len(df_customers):,} rows")

In [ ]:
df_revenue["purchase_date"] = pd.to_datetime(df_revenue["purchase_date"])
df_revenue.head()

In [ ]:
df_products

In [ ]:
df_customers.head()

## 3. Revenue Per Day — Line Chart

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(
    df_revenue["purchase_date"],
    df_revenue["total_revenue"],
    color="steelblue",
    linewidth=1.4,
    label="Daily Revenue",
)

# Rolling 7-day average
rolling = df_revenue["total_revenue"].rolling(7, center=True).mean()
ax.plot(
    df_revenue["purchase_date"],
    rolling,
    color="tomato",
    linewidth=2.2,
    linestyle="--",
    label="7-day Rolling Avg",
)

ax.set_title("Daily Revenue (from 500k synthetic orders)", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Total Revenue ($)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend()
plt.tight_layout()
plt.savefig("revenue_per_day.png", bbox_inches="tight")
plt.show()
print("Saved: revenue_per_day.png")

## 4. Top Products by Revenue — Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

colors = sns.color_palette("muted", len(df_products))
bars = ax.barh(
    df_products["product_category"],
    df_products["total_revenue"],
    color=colors,
    edgecolor="white",
)

# Value labels
for bar in bars:
    width = bar.get_width()
    ax.text(
        width * 1.005,
        bar.get_y() + bar.get_height() / 2,
        f"${width:,.0f}",
        va="center",
        fontsize=10,
    )

ax.set_title("Top Products by Total Revenue", fontsize=14, fontweight="bold")
ax.set_xlabel("Total Revenue ($)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
plt.tight_layout()
plt.savefig("top_products.png", bbox_inches="tight")
plt.show()
print("Saved: top_products.png")

## 5. Customer Segmentation

In [ ]:
# Segment by membership type
seg_membership = (
    df_customers.groupby("membership_type")["customer_id"]
    .count()
    .reset_index()
    .rename(columns={"customer_id": "count"})
    .sort_values("count", ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
axes[0].pie(
    seg_membership["count"],
    labels=seg_membership["membership_type"],
    autopct="%1.1f%%",
    startangle=140,
    colors=sns.color_palette("pastel"),
)
axes[0].set_title("Customer Count by Membership", fontsize=12, fontweight="bold")

# Average spend per membership
seg_spend = (
    df_customers.groupby("membership_type")["avg_spend"]
    .mean()
    .reset_index()
    .sort_values("avg_spend", ascending=True)
)
sns.barplot(
    data=seg_spend,
    y="membership_type",
    x="avg_spend",
    palette="muted",
    ax=axes[1],
)
axes[1].set_title("Avg Spend per Membership Tier", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Avg Spend ($)")
axes[1].set_ylabel("")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

plt.tight_layout()
plt.savefig("customer_segmentation.png", bbox_inches="tight")
plt.show()
print("Saved: customer_segmentation.png")

## 6. Satisfaction Score Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

sns.histplot(
    data=df_customers,
    x="avg_satisfaction",
    bins=20,
    hue="membership_type",
    multiple="stack",
    palette="muted",
    ax=ax,
)

ax.set_title("Satisfaction Score Distribution by Membership", fontsize=13, fontweight="bold")
ax.set_xlabel("Avg Satisfaction Score")
ax.set_ylabel("Customer Count")
plt.tight_layout()
plt.savefig("satisfaction_distribution.png", bbox_inches="tight")
plt.show()
print("Saved: satisfaction_distribution.png")

## Summary

| Chart | File |
|-------|------|
| Daily Revenue + 7-day Rolling Avg | `revenue_per_day.png` |
| Top Products by Revenue | `top_products.png` |
| Customer Segmentation (count + avg spend) | `customer_segmentation.png` |
| Satisfaction Distribution | `satisfaction_distribution.png` |